In [64]:
import pandas as pd
import numpy as np
import nltk
import itertools
import time
import emoji
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics
import matplotlib.pyplot as plt
nltk.download('stopwords')
nltk.download('vader_lexicon')
start = time. time()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [133]:
#Read the data set from location
def ImportFile(fileName):
    df = pd.read_csv(fileName, index_col=None)
    # Remove rows where all values are NaN
    df = df.dropna(how="all")
    return df

In [134]:
#Prepare Training and Test Set
def split_train_test(indep_X, dep_Y):
    X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
    return X_train, X_test, y_train, y_test

In [135]:
#Method for TfidfVectorizer
def TfidfVectorizer(X_train, X_test):
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer
    tfidf_Vectorizer = TfidfVectorizer(stop_words='english')
    counter_Train = tfidf_Vectorizer.fit_transform(X_train)
    counter_Train_dense = counter_Train.toarray()
    #print(counter_Train)
    #print(X_test)
    count_test = tfidf_Vectorizer.transform(X_test)
    count_test_dense = count_test.toarray()
    #print(count_test)
    print(len(tfidf_Vectorizer.get_feature_names_out()))
    return counter_Train, count_test, tfidf_Vectorizer, counter_Train_dense, count_test_dense

In [136]:
#Method for Unique words count calculation
# Get the number of unique features (words) learned by CountVectorizer
# - count_vectorizer.get_feature_names_out(): returns an array of all words in the vocabulary
#   → This equals the number of features (columns) in your training matrix
def UniqueWordsCount(tfidf_Vectorizer):
    print(len(tfidf_Vectorizer.get_feature_names_out()))
    return(len(tfidf_Vectorizer.get_feature_names_out()))

In [137]:
# Definintion MultinomialNB() classification model
def MultinomialNB_Classification(count_train, y_train, count_test, y_test):
    clf = MultinomialNB()
    clf.fit(count_train, y_train)
    pred = clf.predict(count_test)
    score = metrics.accuracy_score(y_test, pred)
    print("accuracy:   %0.3f" % score)
    cm = metrics.confusion_matrix(y_test, pred, labels=['Flirt', 'Non-Flirt'])
    return score, cm, y_test, pred, clf

In [138]:
def classification_report(y_test, pred):
    from sklearn.metrics import classification_report
    report=classification_report(y_test, pred)
    return report

In [139]:
def give_emoji_free_text(text):
    allchars = [str for str in text.decode('utf-8')]
    emoji_list = [c for c in allchars if c in emoji.EMOJI_DATA]
    clean_text = ' '.join([str for str in text.decode('utf-8').split() if not any(i in str for i in emoji_list)])
    return clean_text

In [140]:
def deEmojify(inputString):
    return inputString.encode('ascii', 'ignore').decode('ascii')

In [163]:
def Wholeprocess(df):
    result = []
    result1 = []
    df = df.dropna(subset=['Date', 'Time', 'Name','Text','Label'])
    df['Text'] = df['Text'].apply(lambda x : give_emoji_free_text(str(x).encode('utf8')))
    df['Name'] = df['Name'].apply(lambda x : deEmojify(str(x).lower()))
    df['TotalWordCount'] = df['Text'].str.split().str.len()
    df.index = range(df.shape[0])
    #Total words in chat
    total_word_file = df['TotalWordCount'].sum()
    "Finding Top 3 chats with words count"
    chater = df['Name'].value_counts().head(7).to_dict()
    print(chater)
    Talker = df['Name'].value_counts().idxmax()
    Talker1 = Talker.upper()
    print("\n More Talkative:", Talker1)
    result.append(Talker1)
    Less_Talker=df['Name'].value_counts().idxmin()
    Less_Talker1=Less_Talker.upper()
    print("\n Less Talktative:", Less_Talker.upper())
    result.append(Less_Talker1)
    Talker_chat = pd.DataFrame(df[df.Name==Talker])
    Less_chat = pd.DataFrame(df[df.Name==Less_Talker])
    unique_Frequency_Talker= pd.DataFrame(Talker_chat['Text'].str.split(' ', expand = True).stack().value_counts())
    unique_Frequency_Lesser= pd.DataFrame(Less_chat['Text'].str.split(' ', expand = True).stack().value_counts())
    print("\n Unique Frequency Talker\n")
    print(unique_Frequency_Talker.head())
    unique_Frequency_Talker['Usage of word'] = (unique_Frequency_Talker['count'] / Talker_chat['TotalWordCount'].sum())*100
    unique_Frequency_Lesser['Usage of word'] = (unique_Frequency_Talker['count'] / Talker_chat['TotalWordCount'].sum())*100
    #print(unique_Frequency_Talker)
    print("\n Most no of times used words")
    Most_Frequnetly_Used_Words = pd.DataFrame(unique_Frequency_Talker[unique_Frequency_Talker['count'] > 10])
    print(Most_Frequnetly_Used_Words)
    return result, result1, df

In [159]:
def DateAndTimeAnalysis(df):
    import re
    df['Time'] = df['Time'].apply(lambda x: re.sub(r'[^\x00-\x7F]+', ' ', x).strip())
    most_active_date = df["Date"].value_counts().idxmax()
    most_active_time = df["Time"].value_counts().idxmax()
    print("\nMost Active Day, Most Active Time")
    print(most_active_date)
    print(most_active_time)
    return df

In [143]:
df = ImportFile("Flirt_Dataset.csv")

In [144]:
print(df.head(5))

         Date        Time              Name  \
0  12-12-2024    3:22 pm    DSMax_HelloKids   
1  12-12-2024    4:08 pm          AnandAmmu   
2  01-01-2025    9:01 pm          AnandAmmu   
3  01-01-2025    9:02 pm          AnandAmmu   
4  01-01-2025   10:55 pm    DSMax_HelloKids   

                                         Text      Label  
0  Plz send me santhosh mother  mobile number  Non-Flirt  
1                                  Pavithra J  Non-Flirt  
2                                      Hi Mam  Non-Flirt  
3                     Also please let us know  Non-Flirt  
4                                      Ok Mam  Non-Flirt  


In [145]:
print(df.columns)
indep_X = df['Text']
dep_Y = df['Label']
X_train, X_test, y_train, y_test = split_train_test(indep_X, dep_Y)

Index(['Date', 'Time', 'Name', 'Text', 'Label'], dtype='object')


In [146]:
counter_Train, count_test, tfidf_Vectorizer, counter_Train_dense, count_test_dense = TfidfVectorizer(X_train, X_test)

234


In [147]:
UniqueWordsCount(tfidf_Vectorizer)

234


234

In [148]:
score, cm, y_test, pred, clf = MultinomialNB_Classification(counter_Train, y_train, count_test, y_test)

accuracy:   1.000


In [149]:
report=classification_report(y_test, pred)
print(report)

              precision    recall  f1-score   support

       Flirt       1.00      1.00      1.00        25
   Non-Flirt       1.00      1.00      1.00       112

    accuracy                           1.00       137
   macro avg       1.00      1.00      1.00       137
weighted avg       1.00      1.00      1.00       137



In [150]:
df["Text"][0]

'Plz send me santhosh mother  mobile number'

In [151]:
counter_Train[[0]]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2 stored elements and shape (1, 234)>

In [152]:
print(X_train[0])

Plz send me santhosh mother  mobile number


In [153]:
clf.predict(counter_Train[[0]])

array(['Non-Flirt'], dtype='<U9')

In [154]:
clf.predict(counter_Train[[400]])

array(['Flirt'], dtype='<U9')

In [155]:
counter_Train.shape

(409, 234)

In [164]:
result, result1, df = Wholeprocess(df)

{' dsmax_hellokids': 426, 'teacher': 98, ' anandammu': 22}

 More Talkative:  DSMAX_HELLOKIDS

 Less Talktative:  ANANDAMMU

 Unique Frequency Talker

          count
Dear        254
Parents     215
gentle       47
A            47
reminder     43

 Most no of times used words
          count  Usage of word
Dear        254      21.026490
Parents     215      17.798013
gentle       47       3.890728
A            47       3.890728
reminder     43       3.559603
This         30       2.483444
message      30       2.483444
was          29       2.400662
deleted      29       2.400662
parents      24       1.986755
parent       16       1.324503
for          15       1.241722
the          13       1.076159
Ok           11       0.910596


In [160]:
df = DateAndTimeAnalysis(df)


Most Active Day, Most Active Time
21-08-2025
5:54 pm
